Importando bibliotecas e carregando dados tratados

In [1]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, roc_auc_score, f1_score

df = pd.read_csv("../data/telco_churn_clean.csv")

Carregando apenas as colunas que iremos usar

In [ ]:
drop_cols = ['CustomerID', 'Country', 'State', 'City', 'Lat Long', 
             'Churn Reason', 'Churn Score', 'Churn Value', 'CLTV']

X = df.drop(columns=drop_cols + ['Churn Label'])
# ALVO - transformando 1 ou 0
y = (df['Churn Label'] == 'Yes').astype(int)

# FEATURES - transformando em dummies ( text em numeros )
X = pd.get_dummies(X)

Treino e avaliação dos baselines

In [3]:
scoring = {
    'roc_auc': 'roc_auc',
    'f1': make_scorer(f1_score)
}

dummy = DummyClassifier(strategy='most_frequent')
logreg = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])

for nome, modelo in [('Dummy', dummy), ('Logistic Regression', logreg)]:
    scores = cross_validate(modelo, X, y, cv=StratifiedKFold(5), scoring=scoring)
    print(f"{nome} — AUC: {scores['test_roc_auc'].mean():.3f} | F1: {scores['test_f1'].mean():.3f}")

Dummy — AUC: 0.500 | F1: 0.000
Logistic Regression — AUC: 0.856 | F1: 0.619


Iniciando MLflow

In [6]:
import mlflow

mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("telco-churn-baselines")

for nome, modelo in [('Dummy', dummy), ('Logistic Regression', logreg)]:
    scores = cross_validate(modelo, X, y, cv=StratifiedKFold(5), scoring=scoring)
    
    with mlflow.start_run(run_name=nome):
        mlflow.log_param("model", nome)
        mlflow.log_metric("auc_roc", scores['test_roc_auc'].mean())
        mlflow.log_metric("f1", scores['test_f1'].mean())
        print(f"{nome} — AUC: {scores['test_roc_auc'].mean():.3f} | F1: {scores['test_f1'].mean():.3f}")

c:\Users\mateus.nascimento\telco-churn\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/22 13:22:57 INFO mlflow.tracking.fluent: Experiment with name 'telco-churn-baselines' does not exist. Creating a new experiment.


Dummy — AUC: 0.500 | F1: 0.000
Logistic Regression — AUC: 0.856 | F1: 0.619
